# 01 Qwen2.5-7B-Instruct
Zero-shot Text-to-SQL on Spider dev with `qwen/qwen-2.5-7b-instruct` (temperature 0, 512 tokens).

In [5]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display
from common import config, env, data, preflight
from common.config import RunConfig, resolve_limit, load_api_key
from common.llm_client import OpenRouterClient, fetch_endpoints
from common.pipeline import run_model_pipeline

In [6]:
# Run_Config (single source of truth for this notebook)
RUN_NAME = "qwen25_7b"
MODEL_ID = "qwen/qwen-2.5-7b-instruct"
LIMIT = resolve_limit(None)  # None = full dev set; or set A1_LIMIT
PROVIDER_ORDER = ("Phala",)  # pinned so one quantization serves every call; see the fetch_endpoints table below

def make_cfg():
    return RunConfig(run_name=RUN_NAME, model_id=MODEL_ID, temperature=0.0, max_tokens=512,
                     reasoning_enabled=False, provider_order=tuple(PROVIDER_ORDER),
                     limit=LIMIT)

cfg = make_cfg()
cfg.to_dict()

{'run_name': 'qwen25_7b',
 'model_id': 'qwen/qwen-2.5-7b-instruct',
 'temperature': 0.0,
 'top_p': None,
 'top_k': None,
 'max_tokens': 512,
 'reasoning_enabled': False,
 'provider_order': ['Phala'],
 'allow_fallbacks': False,
 'require_parameters': True,
 'seed': 0,
 'prompt_version': 'v1',
 'threads': 8,
 'limit': None,
 'max_attempts': 6,
 'request_timeout_s': 300.0,
 'preflight_n': 8,
 'retry_on_empty': False,
 'egsc_max_retries': 2,
 'egsc_exec_timeout_s': 5.0,
 'sample_rows': 3,
 'value_trunc': 64}

## Provider endpoints

In [7]:
# Available providers + reported quantization; choose PROVIDER_ORDER from this table.
endpoints = fetch_endpoints(MODEL_ID)
display(pd.DataFrame(endpoints))

,provider_name,quantization,context_length,pricing
0,Phala,unknown,32768,"{'prompt': '0.0000001', 'completion': '0.00000..."


## Run, extract, predict, evaluate, Official_CLI check

In [8]:
client = OpenRouterClient(api_key=load_api_key(required=True))

In [9]:
cfg = make_cfg()
with env.track_run(RUN_NAME, LIMIT, extra={"fingerprint": cfg.fingerprint()}):
    res = run_model_pipeline(cfg, client, endpoints)
display(res.eval.table)
print("missing records:", res.n_missing, "| CLI check passed:", not res.mismatches)

qwen25_7b[full]:   0%|          | 0/1034 [00:00<?, ?it/s]

[pipeline] run: RunStats(n_total=1034, n_skipped=0, n_ok=1034, n_error=0, providers=Counter({'Phala': 1034}), bad_lines=[], stopped_early=False)
[pipeline] wrapper totals match the Official_CLI.


,count,ex,em
easy,248,0.870968,0.520161
medium,446,0.804933,0.282511
hard,174,0.557471,0.224138
extra,166,0.518072,0.066265
all,1034,0.733075,0.294971


missing records: 0 | CLI check passed: True


## Provider report

In [10]:
# Served providers vs. the pin, with reported quantization (Req 12.5, 12.6, 22.5)
display(res.providers)
if (res.providers["flag"] == "outside pin").any():
    print("WARNING: some calls were served by providers outside the pin.")

,provider,calls,share,quantization,in_pin,flag
0,Phala,1034,1.0,unknown,True,
